In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import polars as pl
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/home/aditya/dev/prod-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
# Download dataset from kaggle
# !curl -L -o /content/data/genius-lyrics.zip\
#   https://www.kaggle.com/api/v1/datasets/download/carlosgdcj/genius-song-lyrics-with-language-information
# !unzip /content/data/genius-lyrics.zip -d /content/data/
# !rm /content/data/genius-lyrics.zip

In [72]:
# CSV_PATH = "../data/lyrics/song_lyrics.csv"
CSV_PATH = "../data/lyrics/song_lyrics_the_strokes.csv"
BATCH_SIZE = 10000
CHROMA_PATH = "../db/chroma/genius-lyrics-the-strokes2"
# MILVUS_URI = "http://localhost:19530"

In [62]:

size_gb = os.path.getsize(CSV_PATH) / (1024 ** 3)
print(f"{size_gb:.4f} GB")

0.0001 GB


In [63]:
lf = pl.scan_csv(CSV_PATH)
lf.collect_schema()

Schema([('title', String),
        ('tag', String),
        ('artist', String),
        ('year', Int64),
        ('views', Int64),
        ('features', String),
        ('lyrics', String),
        ('id', Int64),
        ('language_cld3', String),
        ('language_ft', String),
        ('language', String)])

In [64]:
# Number of rows
ROWS = lf.select(pl.len()).collect().item()
ROWS

101

In [65]:
import re

def split_lyrics(lyrics):
    chunks = [
        c.strip()
        for c in re.split(r"\n\n", lyrics)
        if c.strip()
    ]
    return chunks

In [36]:
embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8332.14it/s]


In [37]:
# download milvus script to run it in docker
# !curl -sfL https://raw.githubusercontent.com/milvus-io/milvus/master/scripts/standalone_embed.sh -o ../standalone_embed.sh
# requires sudo -
# !bash standalone_embed.sh start

In [38]:
# from pymilvus import Collection, MilvusException, MilvusClient, connections, db, utility


# connections.connect(
#     alias=client._using,
#     uri=MILVUS_URI,
#     user="root",
#     password="Milvus",
# )

# vectorstore = Milvus(
#     embedding_function=embeddings,
#     collection_name="genius-lyrics",
#     connection_args={"uri": MILVUS_URI},
# )

# try:
#     existing_databases = client.list_databases()
#     if "default" in existing_databases:
#         print(f"Database 'default' already exists.")

#         client.using_database(db_name="default")

#         collections = utility.list_collections()
#         for collection_name in collections:
#             collection = Collection(name=collection_name)
#             collection.drop()
#             print(f"Collection '{collection_name}' has been dropped.")
#     else:
#         # print(f"Database '{db_name}' does not exist.")
#         # client.create_database(db_name)
#         # client.using_database(db_name)
#         # print(f"Database '{db_name}' created successfully.")
#         pass
# except MilvusException as e:
#     print(f"An error occurred: {e}")

In [73]:
vectorstore = Chroma(
    collection_name="genius-lyrics-the-strokes",
    embedding_function=embeddings,
    persist_directory=CHROMA_PATH,

)

In [67]:
from langchain_core.documents import Document

# vectorstore.delete_collection()

In [74]:
from langchain_core.documents import Document
from tqdm import tqdm
from uuid import uuid4


def clean_metadata_value(value):
    if value is None:
        return ""
    if hasattr(value, "__class__") and value.__class__.__name__ == "NAType":
        return ""
    try:
        if value != value:  # NaN check without importing math
            return ""
    except Exception:
        pass
    return value


def build_documents_from_row(row):
    chunks = split_lyrics(row["lyrics"])
    return [
        Document(
            page_content=chunk,
            metadata={
                "chunk_id": i,
                "song_id": clean_metadata_value(row["id"]),
                "title": clean_metadata_value(row["title"]),
                "tag": clean_metadata_value(row["tag"]),
                "artist": clean_metadata_value(row["artist"]),
                "year": clean_metadata_value(row["year"]),
                "language": clean_metadata_value(row["language"]),
            },
        )
        for i, chunk in enumerate(chunks)
    ]


pbar = tqdm(total=ROWS, desc="Processing songs")
chunks_inserted = 0
songs_processed = 0
buffer: list[Document] = []

batches = lf.collect_batches(chunk_size=BATCH_SIZE)
for batch in batches:
    for row in batch.iter_rows(named=True):
        buffer.extend(build_documents_from_row(row))

        if len(buffer) >= BATCH_SIZE:
            vectorstore.add_documents(buffer)
            chunks_inserted += len(buffer)
            pbar.set_postfix(chunks=chunks_inserted)
            buffer = []

        songs_processed += 1
        pbar.update(1)

    if buffer:
        uuids = [str(uuid4()) for _ in range(len(buffer))]
        vectorstore.add_documents(buffer, ids=uuids)
        chunks_inserted += len(buffer)
        pbar.set_postfix(chunks=chunks_inserted)
        buffer = []

    # break  # remove this line to ingest the full dataset

pbar.close()

Processing songs: 100%|██████████| 101/101 [00:08<00:00, 12.53it/s, chunks=573]


In [75]:
vectorstore.similarity_search("door", k=20)

[Document(id='7c0a7310-019d-49b0-b715-321f97cda494', metadata={'song_id': 5244673, 'title': 'At the Door', 'artist': 'The Strokes', 'tag': 'pop', 'chunk_id': 1, 'language': 'en', 'year': 2020}, page_content='[Pre-Chorus]\nArrive at the door\nAnyone home?\nHave\u205fI\u205flost\u205fit all?'),
 Document(id='8170d19f-d595-4401-89f5-a770ed679a1b', metadata={'language': 'en', 'year': 2013, 'song_id': 131394, 'chunk_id': 1, 'artist': 'The Strokes', 'title': '80s Comedown Machine', 'tag': 'rock'}, page_content="Tried to get in with you\nFor a second time\nYou said the world's closed in\nBut you crawled in\nBack inside"),
 Document(id='e125a6f5-5c8f-4dca-ac85-9102c9678cc7', metadata={'tag': 'rock', 'title': 'Not the Same Anymore', 'language': 'en', 'year': 2020, 'artist': 'The Strokes', 'song_id': 5244588, 'chunk_id': 0}, page_content="[Verse 1]\nYou're not the same anymore\nDon’t wanna play that game anymore\nYou'd make a better window than\u2005a\u2005door\nOh, the strangers,\u2005they impl